In [1]:
#Data Cleaning
import pandas as pd

geo = pd.read_csv('olist_geolocation_dataset.csv')
print("Shape:", geo.shape)
print("Nulls:\n", geo.isnull().sum())
print("Duplicates:", geo.duplicated().sum())

Shape: (1000163, 5)
Nulls:
 geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Duplicates: 261831


In [2]:
# Remove duplicate rows
geo_clean = geo.drop_duplicates()
print("Rows after removing duplicates:", len(geo_clean))

Rows after removing duplicates: 738332


In [3]:
#zip code
geo_unique = geo_clean.drop_duplicates(subset=['geolocation_zip_code_prefix'], keep='first')
print("Unique zip codes:", len(geo_unique))

Unique zip codes: 19015


In [4]:
#Verify the result
print("Nulls after cleaning:\n", geo_unique.isnull().sum())
print("Duplicates after cleaning:", geo_unique.duplicated().sum())
print("Final shape:", geo_unique.shape)

Nulls after cleaning:
 geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Duplicates after cleaning: 0
Final shape: (19015, 5)


## Task 2 — Normalization of Product Category Names

In [6]:
import pandas as pd

# Load the products file
products = pd.read_csv('olist_products_dataset.csv')

# Load the translation file
translation = pd.read_csv('product_category_name_translation.csv')

In [7]:
# See the first 5 rows of products
print(products.head())

                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   
3                 27.0                       261.0                 1.0   
4                 37.0                       402.0                 4.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0               16.0               10.0              14.0  
1            1000.0       

In [8]:
# See the first 5 rows of translation
print(translation.head())

    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor


In [9]:
#Understand the size of the problem
# How many rows does products have?
print("Total products:", len(products))

# How many category names are missing?
print("Missing category names:", products['product_category_name'].isnull().sum())

# How many unique Portuguese categories exist?
print("Unique categories in products:", products['product_category_name'].nunique())

# How many translations do we have?
print("Translations available:", len(translation))

Total products: 32951
Missing category names: 610
Unique categories in products: 73
Translations available: 71


In [11]:
# Merge the two files together
# Attach English names to the products table
products_merged = products.merge(translation, on='product_category_name', how='left')

# See the result
print(products_merged[['product_category_name', 'product_category_name_english']].head(10))

   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares
5  instrumentos_musicais           musical_instruments
6             cool_stuff                    cool_stuff
7       moveis_decoracao               furniture_decor
8       eletrodomesticos               home_appliances
9             brinquedos                          toys


In [12]:
#Find what didn't translate
# Filter rows where English name is still missing
unmatched = products_merged[products_merged['product_category_name_english'].isnull()]

# Show the Portuguese names that failed to translate
print(unmatched['product_category_name'].value_counts())


product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


In [13]:
#Manually add the missing translations
# Create a dictionary of the 2 missing translations
manual_map = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_food_preparers'
}

In [14]:
# Fill missing English names using our manual map
products_merged['product_category_name_english'] = products_merged.apply(
    lambda row: manual_map.get(row['product_category_name'], row['product_category_name_english'])
    if pd.isnull(row['product_category_name_english']) else row['product_category_name_english'],
    axis=1
)

In [15]:
print(products_merged[products_merged['product_category_name'] == 'pc_gamer']
      [['product_category_name', 'product_category_name_english']].head(3))

      product_category_name product_category_name_english
1628               pc_gamer                      pc_gamer
7478               pc_gamer                      pc_gamer
16930              pc_gamer                      pc_gamer


In [16]:
# Make all English category names lowercase, no spaces, no extra whitespace
products_merged['product_category_name_english'] = (
    products_merged['product_category_name_english']
    .str.lower()          # converts to lowercase: Health_Beauty → health_beauty
    .str.strip()          # removes spaces from the edges: ' health_beauty ' → 'health_beauty'
    .str.replace(' ', '_', regex=False)  # replaces spaces with underscores
)

In [17]:
# Final verification
# How many nulls remain in the English column?
print("Remaining nulls:", products_merged['product_category_name_english'].isnull().sum())

Remaining nulls: 610


In [18]:
# See a clean sample of Portuguese → English mapping
print(products_merged[['product_category_name', 'product_category_name_english']]
      .drop_duplicates()
      .head(15))

               product_category_name product_category_name_english
0                         perfumaria                     perfumery
1                              artes                           art
2                      esporte_lazer                sports_leisure
3                              bebes                          baby
4              utilidades_domesticas                    housewares
5              instrumentos_musicais           musical_instruments
6                         cool_stuff                    cool_stuff
7                   moveis_decoracao               furniture_decor
8                   eletrodomesticos               home_appliances
9                         brinquedos                          toys
10                   cama_mesa_banho                bed_bath_table
14  construcao_ferramentas_seguranca     construction_tools_safety
17            informatica_acessorios         computers_accessories
22                      beleza_saude                 health_be

In [19]:
# How many unique English categories do we end up with?
print("Unique English categories:", products_merged['product_category_name_english'].nunique())

Unique English categories: 73


# Task 3 — Final Clean CSV Export for SQL Import

In [20]:
geo_unique.to_csv('geo_clean.csv', index=False)
print("Exported geo_clean.csv:", geo_unique.shape)

Exported geo_clean.csv: (19015, 5)


In [21]:
products_merged.to_csv('products_clean.csv', index=False)
print("Exported products_clean.csv:", products_merged.shape)

Exported products_clean.csv: (32951, 10)


In [23]:
customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')

customers.to_csv('customers_clean.csv', index=False)
orders.to_csv('orders_clean.csv', index=False)
order_items.to_csv('order_items_clean.csv', index=False)

print("All clean CSVs exported.")

All clean CSVs exported.


In [24]:
for fname in ['customers_clean.csv', 'orders_clean.csv', 'order_items_clean.csv',
              'geo_clean.csv', 'products_clean.csv']:
    df = pd.read_csv(fname)
    print(f"{fname}: {df.shape} | nulls: {df.isnull().sum().sum()} total | dupes: {df.duplicated().sum()}")

customers_clean.csv: (99441, 5) | nulls: 0 total | dupes: 0
orders_clean.csv: (99441, 8) | nulls: 4908 total | dupes: 0
order_items_clean.csv: (112650, 7) | nulls: 0 total | dupes: 0
geo_clean.csv: (19015, 5) | nulls: 0 total | dupes: 0
products_clean.csv: (32951, 10) | nulls: 3058 total | dupes: 0
